## Лабораторная работа №1. Интерактивный анализ данных велопарковок SF Bay Area Bike Share в Apache Spark.
## Выполнил студент группы 6401-010302D, Стрельников Никита

## Описание данных

https://www.kaggle.com/benhamner/sf-bay-area-bike-share

stations.csv схема:

```
id: station ID number
name: name of station
lat: latitude
long: longitude
dock_count: number of total docks at station
city: city (San Francisco, Redwood City, Palo Alto, Mountain View, San Jose)
installation_date: original date that station was installed. If station was moved, it is noted below.
```

trips.csv схема:

```
id: numeric ID of bike trip
duration: time of trip in seconds
start_date: start date of trip with date and time, in PST
start_station_name: station name of start station
start_station_id: numeric reference for start station
end_date: end date of trip with date and time, in PST
end_station_name: station name for end station
end_station_id: numeric reference for end station
bike_id: ID of bike used
subscription_type: Subscriber = annual or 30-day member; Customer = 24-hour or 3-day member
zip_code: Home zip code of subscriber (customers can choose to manually enter zip at kiosk however data is unreliable)
```

Импорты

In [ ]:
import itertools

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType 

from geopy.distance import geodesic

In [43]:
spark = (
    SparkSession
        .builder
        .master("local[*]")
        .appName("BikeShareLab")
        .getOrCreate()
)

### 0. Загрузка данных

In [44]:
stations = (
    spark
        .read
        .option("header", True)
        .option("inferSchema", True)
        .csv("stations.csv")
)

trips = (
    spark
        .read
        .option("header", True)
        .option("inferSchema", True)
        .csv("trips.csv")
)

In [45]:
stations.printSchema()
trips.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- dock_count: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- installation_date: string (nullable = true)

root
 |-- id: integer (nullable = true)
 |-- duration: integer (nullable = true)
 |-- start_date: string (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: integer (nullable = true)
 |-- end_date: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: integer (nullable = true)
 |-- bike_id: integer (nullable = true)
 |-- subscription_type: string (nullable = true)
 |-- zip_code: string (nullable = true)



### 1. Найти велосипед с максимальным временем пробега

In [46]:
# Группируем данные по bike_id и вычисляем длительность максимальной поездки
bike_max_duration = (
    trips
        .groupBy("bike_id")
        .agg(F.sum("duration").alias("total_duration_sec"))
        .orderBy(F.desc("total_duration_sec"))
)

bike_max_duration.show(1, truncate=False)

+-------+------------------+
|bike_id|total_duration_sec|
+-------+------------------+
|535    |18611693          |
+-------+------------------+
only showing top 1 row


### 2. Найти наибольшее геодезическое расстояние между станциями

In [51]:
stations_columns = stations.select("id", "name", "lat", "long").collect()

max_pair = None
max_distance = -1

# Проходим по всем комбинациям
for s1, s2 in itertools.combinations(stations_columns, 2):
    # Для подсчета геодезического расстояния используем функцию geodesic
    dist = geodesic((s1["lat"], s1["long"]), (s2["lat"], s2["long"])).kilometers
    
    if dist > max_distance:
        # Обновляем пары и расстояние между ними
        max_distance = dist
        max_pair = (
            s1["id"], s1["name"],
            s2["id"], s2["name"],
            dist
        )

print(f"station1_id = {max_pair[0]}")
print(f"station1_name = {max_pair[1]}")
print(f"station2_id = {max_pair[2]}")
print(f"station2_name = {max_pair[3]}")
print(f"distance_km = {max_pair[4]:.3f}")

station1_id = 16
station1_name = SJSU - San Salvador at 9th
station2_id = 60
station2_name = Embarcadero at Sansome
distance_km = 69.921


### 3. Найти путь велосипеда с максимальным временем пробега через станции

Из 1 пункта мы знаем id велосипедиста. Теперь мы можем просто вывести его путь.

In [33]:
top_bike_id = 535

# Выбираем только необходимые атрибуты
top_bike_path_compact = (
    trips
        .filter(F.col("bike_id") == top_bike_id) # Выбор известного велосипедиста
        .select(
            "id",
            "bike_id",
            "start_station_id",
            "end_station_id"
        )
        .orderBy(F.col("id").cast(IntegerType()))
)

# По умолчанию выводим только 50 первых
top_bike_path_compact.show(50, truncate=False)
# top_bike_path_compact.show(top_bike_path_compact.count(), truncate=False)

+-----+-------+----------------+--------------+
|id   |bike_id|start_station_id|end_station_id|
+-----+-------+----------------+--------------+
|4966 |535    |47              |70            |
|5067 |535    |70              |69            |
|5179 |535    |69              |77            |
|5199 |535    |77              |64            |
|7806 |535    |61              |42            |
|11422|535    |58              |72            |
|12245|535    |72              |47            |
|12485|535    |47              |60            |
|12558|535    |60              |46            |
|13107|535    |46              |77            |
|13423|535    |77              |77            |
|14380|535    |77              |62            |
|14581|535    |62              |61            |
|15231|535    |55              |61            |
|15242|535    |61              |60            |
|15347|535    |60              |41            |
|15605|535    |41              |50            |
|15611|535    |50              |41      

### 4. Найти количество велосипедов в системе

In [34]:
bike_count = trips.select("bike_id").distinct().count()
print("Количество велосипедов в системе:", bike_count)

Количество велосипедов в системе: 700


### 5. Найти пользователей, потративших на поездки более 3 часов

In [35]:
# Группировка данных по zip_code и вычисление максимальной продолжительности поездки в секундах
users_over_3h = (
    trips
        .filter(F.col("zip_code").isNotNull())
        .groupBy("zip_code")
        .agg(
            F.sum("duration").alias("total_duration_sec"),
            F.round(F.sum("duration") / 3600, 2).alias("total_duration_hours"),
            F.count("*").alias("trip_count")
        )
        .filter(F.col("total_duration_sec") > 10800) # Оставляем только тех, у кого длительность поездки строго больше 3 часов (10800 секунд)
        .orderBy(F.desc("total_duration_sec")) # Сортировка по убыванию
)

users_over_3h.show(truncate=False)

+--------+------------------+--------------------+----------+
|zip_code|total_duration_sec|total_duration_hours|trip_count|
+--------+------------------+--------------------+----------+
|94107   |49757162          |13821.43            |78704     |
|nil     |45725550          |12701.54            |10682     |
|94105   |25596128          |7110.04             |42672     |
|94133   |21637675          |6010.47             |31359     |
|94102   |19128021          |5313.34             |19757     |
|94103   |19127388          |5313.16             |26673     |
|95531   |17270400          |4797.33             |1         |
|94111   |14244997          |3956.94             |21409     |
|95112   |12742370          |3539.55             |11564     |
|94109   |12057128          |3349.2              |13989     |
|94040   |7807926           |2168.87             |7114      |
|94110   |7421936           |2061.65             |7621      |
|94117   |6901313           |1917.03             |9851      |
|94301  

In [38]:
spark.stop()